# Estrategia 1: U-Net + EfficientNet-B0
## Segmentación Semántica Multiclase (T1-L5)

**Meta:** Superar Dice por vértebra de 0.74 del antecedente [16].

## 0. Setup

In [2]:
%matplotlib inline
#!pip install -q segmentation-models-pytorch albumentations

import sys
sys.path.insert(0, '/Users/becavas/data/spine_seg')

import os, torch, numpy as np, matplotlib.pyplot as plt
print('Setup OK')

Setup OK


## 1. Hiperparámetros del experimento

In [ ]:
# Imprimir TODOS los hiperparámetros del experimento
from configs.config import *
import torch

print("=" * 70)
print("CONFIGURACIÓN DEL EXPERIMENTO")
print("=" * 70)
print(f"\n📁 DATASET:")
print(f"  Root: {DATASET_ROOT}")
print(f"  Imágenes excluidas: {IMAGES_TO_EXCLUDE}")
print(f"  Clases: {NUM_CLASSES} (17 vértebras T1-L5 + fondo)")
print(f"  Vértebras: {ANATOMICAL_VERTEBRAE}")
print(f"  Clases excluidas: {len(CLASSES_TO_EXCLUDE)} (cervicales + entidades)")
print(f"\n📐 SPLIT:")
print(f"  Train/Val/Test: {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")
print(f"  Seed: {RANDOM_SEED}")
print(f"\n🔧 PREPROCESAMIENTO:")
print(f"  CLAHE: clip={CLAHE_CLIP_LIMIT}, grid={CLAHE_TILE_GRID}")
print(f"  Normalización: mean={PIXEL_MEAN}, std={PIXEL_STD}")
print(f"\n🎲 AUGMENTATION:")
print(f"  Rotación: ±{AUG_ROTATION_LIMIT}°")
print(f"  Shift: ±{AUG_SHIFT_LIMIT}")
print(f"  Scale: {AUG_SCALE_LIMIT}")
print(f"  Brightness: ±{AUG_BRIGHTNESS_LIMIT}, Contrast: ±{AUG_CONTRAST_LIMIT}")
print(f"  Random crop: {AUG_CROP_MIN*100:.0f}-{AUG_CROP_MAX*100:.0f}% (prob={AUG_CROP_PROB})")
print(f"\n⚙️ ENTRENAMIENTO:")
print(f"  Épocas: {MAX_EPOCHS}, Early stopping: {EARLY_STOPPING_PATIENCE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  LR: {LEARNING_RATE}, Weight decay: {WEIGHT_DECAY}")
print(f"  Warmup: {WARMUP_EPOCHS} épocas")
print(f"  AMP: {USE_AMP}")
print(f"\n⚖️ PÉRDIDA:")
print(f"  Dice weight: {DICE_WEIGHT}, CE weight: {CE_WEIGHT}")
print(f"  Boundary loss weight: {BOUNDARY_LOSS_WEIGHT}")
print(f"  Medianas de área (top-3): { {k: v for k, v in sorted(AREA_MEDIANS.items(), key=lambda x: -x[1])[:3]} }")
print(f"  Medianas de área (bottom-3): { {k: v for k, v in sorted(AREA_MEDIANS.items(), key=lambda x: x[1])[:3]} }")
print(f"\n🖥️ HARDWARE:")
device = 'cuda' if torch.cuda.is_available() else ('mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')
print(f"  Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name()}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"\n📂 OUTPUT:")
print(f"  Run: {CURRENT_RUN_DIR}")


# Específicos de U-Net
print(f'\n🏗️ MODELO:')
print(f'  Arquitectura: U-Net + EfficientNet-B0')
print(f'  Encoder weights: ImageNet')
print(f'  Encoder LR factor: 0.1 (10x menor que decoder)')
print(f'  Input size: {IMG_SIZES["unet"]}')
print(f'  Clases de salida: {NUM_CLASSES}')

## 2. Carga de datos y split

In [ ]:
# Carga de datos y split (IDÉNTICO para las 3 estrategias)
from shared.data_loader import (
    load_dataset_index, build_label_mapping,
    load_cobb_metrics, stratified_split
)

df_index, original_id_to_name = load_dataset_index()
label_mapping = build_label_mapping(original_id_to_name)
df_train, df_val, df_test = stratified_split(df_index)
df_cobb = load_cobb_metrics()

# Verificación rápida
print(f"\n📊 Verificación de clases en máscara de ejemplo:")
import cv2, numpy as np
from shared.data_loader import get_mask_multiclass_path
sample_row = df_train.iloc[0]
mask_path = get_mask_multiclass_path(sample_row)
mask_raw = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
print(f"  Imagen: {sample_row['image']}")
print(f"  Máscara dtype: {mask_raw.dtype}, shape: {mask_raw.shape}")
print(f"  Valores únicos raw: {np.unique(mask_raw).tolist()}")
print(f"  Valores después de remapeo: {sorted(set(label_mapping.get(v, 0) for v in np.unique(mask_raw)))}")


## 3. Verificación del dataset

In [ ]:
%matplotlib inline
from shared.dataset import SpineSegDataset, create_dataloaders
from configs.config import IMG_SIZE_SEMANTIC, PIXEL_MEAN, PIXEL_STD, CLASS_ID_TO_VERTEBRA, NUM_CLASSES

check_ds = SpineSegDataset(df_train.head(5), label_mapping, mode='val', target_size=IMG_SIZE_SEMANTIC)
sample = check_ds[0]
print(f"Imagen: {sample['image'].shape}, rango=[{sample['image'].min():.2f}, {sample['image'].max():.2f}]")
print(f"Máscara: {sample['mask'].shape}, clases={sample['mask'].unique().tolist()}")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i in range(min(3, len(check_ds))):
    s = check_ds[i]
    img = s['image'].permute(1,2,0).numpy() * np.array(PIXEL_STD) + np.array(PIXEL_MEAN)
    axes[0][i].imshow(np.clip(img, 0, 1)); axes[0][i].set_title(f"{s['image_name']}"); axes[0][i].axis('off')
    axes[1][i].imshow(s['mask'].numpy(), cmap='nipy_spectral', vmin=0, vmax=NUM_CLASSES-1)
    axes[1][i].set_title(f"{len(s['mask'].unique())-1} clases"); axes[1][i].axis('off')
plt.tight_layout(); plt.show()
print("✅ Dataset verificado")

## 4. DataLoaders

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    df_train, df_val, df_test, label_mapping,
    batch_size=BATCH_SIZE, num_workers=2, target_size=IMG_SIZE_SEMANTIC
)
batch = next(iter(train_loader))
print(f"Batch: imgs={batch['image'].shape}, masks={batch['mask'].shape}")

## 5. Modelo U-Net

In [ ]:
from models.unet_model import create_unet_model, get_unet_optimizer

model = create_unet_model(encoder_name='efficientnet-b0', encoder_weights='imagenet',
                           num_classes=NUM_CLASSES, in_channels=3)
model.to(device)
with torch.no_grad():
    out = model(torch.randn(1, 3, *IMG_SIZE_SEMANTIC).to(device))
    print(f"Forward OK: output {out.shape} ({NUM_CLASSES} canales)")

## 6. Pérdida y optimizador

In [ ]:
from shared.losses import CombinedLoss, compute_class_weights

class_weights = compute_class_weights(device=device)
print("\nPesos por clase:")
for i in range(NUM_CLASSES):
    name = CLASS_ID_TO_VERTEBRA.get(i, '?')
    print(f"  {name:>12s}: {class_weights[i].item():.3f}")

L4_ID = VERTEBRA_TO_CLASS_ID['L4']
L5_ID = VERTEBRA_TO_CLASS_ID['L5']

criterion = CombinedLoss(class_weights=class_weights, boundary_classes=[L4_ID, L5_ID], device=device)
optimizer = get_unet_optimizer(model, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, encoder_lr_factor=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SCHEDULER_T_MAX - WARMUP_EPOCHS, eta_min=1e-6)

print(f"\nPérdida: Dice({DICE_WEIGHT}) + WCE({CE_WEIGHT}) + Boundary({BOUNDARY_LOSS_WEIGHT}) en L4,L5")

## 7. Entrenamiento

In [ ]:
from shared.trainer import Trainer

trainer = Trainer(model=model, criterion=criterion, optimizer=optimizer,
                  scheduler=scheduler, device=device, experiment_name='unet_effb0')
history = trainer.fit(train_loader, val_loader, max_epochs=MAX_EPOCHS)

In [ ]:
%matplotlib inline
from shared.visualization import plot_training_history
plot_training_history(history, experiment_name='unet_effb0')

## 8. Evaluación detallada en test

In [ ]:
# Evaluación DETALLADA con todas las métricas
summary, test_metrics = trainer.evaluate(test_loader, df_cobb=df_cobb)

# Imprimir tabla completa por clase
print("\n" + "=" * 70)
print("TABLA DETALLADA POR CLASE")
print("=" * 70)
df_class = summary['per_class']
print(f"{'Clase':<12} {'Dice':>8} {'±Std':>8} {'IoU':>8} {'±Std':>8} {'N imgs':>8}")
print("-" * 54)
for _, row in df_class.iterrows():
    if row['class_id'] == 0:
        continue
    print(f"{row['vertebra']:<12} {row['dice_mean']:>8.4f} {row['dice_std']:>8.4f} "
          f"{row['iou_mean']:>8.4f} {row['iou_std']:>8.4f} {row['n_images']:>8.0f}")
print("-" * 54)
print(f"{'PROMEDIO':<12} {summary['dice_vertebra_mean']:>8.4f} {summary['dice_vertebra_std']:>8.4f} "
      f"{summary['iou_vertebra_mean']:>8.4f}")

# Por región
print("\n📍 POR REGIÓN:")
for _, row in summary['per_region'].iterrows():
    print(f"  {row['region']:<25} Dice={row['dice_mean']:.4f}  IoU={row['iou_mean']:.4f}")

# Por tipo
print("\n👤 POR TIPO (Normal vs Escoliosis):")
for _, row in summary['per_type'].iterrows():
    print(f"  {row['type']:<12} Dice={row['dice_mean']:.4f}  (n={row['n_images']:.0f})")

# Por severidad
if 'per_severity' in summary:
    print("\n📏 POR SEVERIDAD:")
    for _, row in summary['per_severity'].iterrows():
        print(f"  {row['severity']:<12} Dice={row['dice_mean']:.4f}  (n={row['n_images']:.0f})")


In [ ]:
%matplotlib inline
from shared.visualization import plot_per_class_metrics
plot_per_class_metrics(summary, experiment_name='unet_effb0')

## 9. Visualización de predicciones

In [ ]:
%matplotlib inline
from shared.visualization import visualize_prediction
from shared.dataset import SpineSegDataset

model.eval()
test_ds = SpineSegDataset(df_test, label_mapping, mode='test', target_size=IMG_SIZE_SEMANTIC)
indices = np.linspace(0, len(test_ds)-1, min(6, len(test_ds)), dtype=int)

for idx in indices:
    s = test_ds[idx]
    with torch.no_grad():
        pred = model(s['image'].unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu().numpy()
    img = s['image'].permute(1,2,0).numpy() * np.array(PIXEL_STD) + np.array(PIXEL_MEAN)
    visualize_prediction(np.clip(img, 0, 1), pred, s['mask'].numpy(),
                         image_name=f"{s['image_name']} ({s['split_type']})")

## 10. Análisis focalizado: L4-L5

In [ ]:
# Análisis focalizado: L4-L5 y regiones problemáticas
from configs.config import VERTEBRA_TO_CLASS_ID

print("=" * 70)
print("ANÁLISIS FOCALIZADO")
print("=" * 70)

df_class = summary['per_class']

# L4-L5 (antecedente: Dice ≈ 0.52)
print("\n🦴 L4-L5 (antecedente Dice ≈ 0.52):")
for v in ['L4', 'L5']:
    cid = VERTEBRA_TO_CLASS_ID[v]
    row = df_class[df_class['class_id'] == cid]
    if len(row) > 0:
        r = row.iloc[0]
        status = '✅' if r['dice_mean'] > 0.52 else '⚠️'
        print(f"  {status} {v}: Dice={r['dice_mean']:.4f} ± {r['dice_std']:.4f} "
              f"IoU={r['iou_mean']:.4f} (n={r['n_images']:.0f})")

# Top-5 mejores y peores
print("\n🏆 Top-5 MEJORES vértebras:")
df_sorted = df_class[df_class['class_id'] > 0].sort_values('dice_mean', ascending=False)
for _, row in df_sorted.head(5).iterrows():
    print(f"  {row['vertebra']}: Dice={row['dice_mean']:.4f}")

print("\n⚠️ Top-5 PEORES vértebras:")
for _, row in df_sorted.tail(5).iterrows():
    print(f"  {row['vertebra']}: Dice={row['dice_mean']:.4f} (n={row['n_images']:.0f})")

# Comparación con baselines
print(f"\n📊 vs BASELINES del antecedente [16]:")
our_dice = summary['dice_vertebra_mean']
print(f"  {'Modelo':<30} {'Dice Vert':>10} {'Δ':>10}")
print(f"  {'-'*50}")
print(f"  {'** NUESTRO **':<30} {our_dice:>10.4f} {'---':>10}")
print(f"  {'YOLOv8m-seg (antecedente)':<30} {'0.7400':>10} {our_dice - 0.74:>+10.4f}")
print(f"  {'Mask R-CNN (antecedente)':<30} {'0.7700':>10} {our_dice - 0.77:>+10.4f}")
print(f"  {'U-Net binaria (antecedente)':<30} {'0.5400':>10} {our_dice - 0.54:>+10.4f}")


## 11. Protocolo de imágenes recortadas

In [ ]:
# Protocolo de imágenes recortadas (20% superior e inferior)
from shared.metrics import MetricsTracker
from shared.dataset import SpineSegDataset

print("=" * 70)
print("PROTOCOLO DE IMÁGENES RECORTADAS")
print("=" * 70)

model.eval()
test_dataset = SpineSegDataset(df_test, label_mapping, mode='test', target_size=IMG_SIZE_SEMANTIC)

crop_results = {}
for crop_name, crop_side in [('sin_recorte', None), ('recorte_sup_20%', 'top'), ('recorte_inf_20%', 'bottom')]:
    metrics_crop = MetricsTracker()
    
    for idx in range(len(test_dataset)):
        sample = test_dataset[idx]
        img = sample['image'].clone()
        mask = sample['mask'].clone()
        
        H = img.shape[1]
        crop_px = int(H * 0.2)
        
        if crop_side == 'top':
            img[:, :crop_px, :] = 0
            mask[:crop_px, :] = 0
        elif crop_side == 'bottom':
            img[:, -crop_px:, :] = 0
            mask[-crop_px:, :] = 0
        
        with torch.no_grad():
            output = model(img.unsqueeze(0).to(device))
            pred = output.argmax(dim=1).squeeze(0).cpu()
        
        metrics_crop.update(pred, mask, sample['image_name'], sample['split_type'])
    
    crop_summary = metrics_crop.compute_summary()
    crop_results[crop_name] = crop_summary
    print(f"\n  {crop_name}: Dice={crop_summary['dice_vertebra_mean']:.4f}")

# Tabla comparativa
print(f"\n{'Condición':<20} {'Dice':>8} {'Δ vs base':>10}")
print("-" * 40)
base_dice = crop_results['sin_recorte']['dice_vertebra_mean']
for name, s in crop_results.items():
    delta = s['dice_vertebra_mean'] - base_dice
    print(f"{name:<20} {s['dice_vertebra_mean']:>8.4f} {delta:>+10.4f}")


## 12. Métricas de eficiencia

In [ ]:
# Métricas de eficiencia
import time

model.eval().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size_mb = total_params * 4 / 1024 / 1024

dummy = torch.randn(1, 3, *IMG_SIZE_SEMANTIC).to(device)
for _ in range(10):
    with torch.no_grad(): _ = model(dummy)
if device == 'cuda': torch.cuda.synchronize()

times_gpu = []
for _ in range(50):
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad(): _ = model(dummy)
    if device == 'cuda': torch.cuda.synchronize()
    times_gpu.append((time.time() - t0) * 1000)

model_cpu = model.cpu()
dummy_cpu = dummy.cpu()
times_cpu = []
for _ in range(10):
    t0 = time.time()
    with torch.no_grad(): _ = model_cpu(dummy_cpu)
    times_cpu.append((time.time() - t0) * 1000)
model.to(device)

print("=" * 70)
print("MÉTRICAS DE EFICIENCIA")
print("=" * 70)
print(f"  Parámetros totales:    {total_params:>12,} ({total_params/1e6:.1f}M)")
print(f"  Parámetros entrenables:{trainable_params:>12,} ({trainable_params/1e6:.1f}M)")
print(f"  Tamaño modelo:         {model_size_mb:>11.1f} MB (float32)")
print(f"  Inferencia GPU:        {np.mean(times_gpu):>11.1f} ± {np.std(times_gpu):.1f} ms")
print(f"  Inferencia CPU:        {np.mean(times_cpu):>11.0f} ± {np.std(times_cpu):.0f} ms")
print(f"  Input size:            {IMG_SIZE_SEMANTIC}")
print(f"\n  Baselines antecedente [16]:")
print(f"  {'Modelo':<25} {'Tamaño':>10} {'GPU ms':>10}")
print(f"  {'-'*45}")
print(f"  {'YOLOv8m-seg':<25} {'52 MB':>10} {'~19':>10}")
print(f"  {'Mask R-CNN':<25} {'335 MB':>10} {'~146':>10}")


## 13. Guardar resultados

In [ ]:
import json

results = {
    'strategy': 'U-Net + EfficientNet-B0',
    'dice_vertebra_mean': float(summary['dice_vertebra_mean']),
    'dice_vertebra_std': float(summary['dice_vertebra_std']),
    'iou_vertebra_mean': float(summary['iou_vertebra_mean']),
    'dice_spine_mean': float(summary['dice_spine_mean']),
    'params_millions': total_params / 1e6,
    'model_size_mb': model_size_mb,
    'inference_gpu_ms': float(np.mean(times_gpu)),
    'inference_cpu_ms': float(np.mean(times_cpu)),
    'per_class': summary['per_class'].to_dict('records'),
    'per_region': summary['per_region'].to_dict('records'),
    'per_type': summary['per_type'].to_dict('records'),
    'crop_impact': {k: float(v['dice_vertebra_mean']) for k, v in crop_results.items()},
    'hyperparams': {
        'encoder': 'efficientnet-b0', 'encoder_weights': 'imagenet',
        'img_size': list(IMG_SIZE_SEMANTIC), 'batch_size': BATCH_SIZE,
        'lr': LEARNING_RATE, 'encoder_lr_factor': 0.1,
        'weight_decay': WEIGHT_DECAY, 'max_epochs': MAX_EPOCHS,
        'loss': f'Dice({DICE_WEIGHT})+WCE({CE_WEIGHT})+Boundary({BOUNDARY_LOSS_WEIGHT})',
        'augmentation': f'crop {AUG_CROP_MIN}-{AUG_CROP_MAX}, rot ±{AUG_ROTATION_LIMIT}°'
    }
}

os.makedirs(str(METRICS_DIR), exist_ok=True)
with open(str(METRICS_DIR / 'results_unet_effb0.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f"Guardado: {METRICS_DIR / 'results_unet_effb0.json'}")

print(f"\n📋 RESUMEN FINAL U-Net:")
print(f"  Dice vértebra: {results['dice_vertebra_mean']:.4f} ± {results['dice_vertebra_std']:.4f}")
print(f"  Dice columna:  {results['dice_spine_mean']:.4f}")
print(f"  Modelo: {results['params_millions']:.1f}M, {results['model_size_mb']:.1f}MB")
print(f"  GPU: {results['inference_gpu_ms']:.1f}ms, CPU: {results['inference_cpu_ms']:.0f}ms")